# 01 — Exploratory Data Analysis

**Owner:** Aparna (Data Lead) · **Support:** Rolando (Data Pipeline & QA)

Sprint-1 EDA over the harmonized DermaFace manifest. Covers **class balance**, **Fitzpatrick skin-type distribution** (the axis our fairness analysis stratifies on), and **image-quality** checks.

> This notebook reads `data/processed/manifest.csv`. If it is missing (e.g. a fresh clone or CI), it auto-generates a small **synthetic smoke slice** so the notebook always runs. On real data the same cells produce the real figures. Clear outputs before committing (`jupyter nbconvert --clear-output`).

In [ ]:
import sys, os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
sns.set_theme(style='whitegrid')

# --- Resolve repo root and make src/ + scripts/ importable ---
ROOT = Path.cwd()
while not (ROOT / 'src' / 'dermaface').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
sys.path.insert(0, str(ROOT / 'scripts'))
from dermaface.config import CLASS_NAMES, FITZPATRICK_TYPES, load_config
print('repo root:', ROOT)

In [ ]:
# --- Load the manifest; build a synthetic slice if none exists ---
cfg = load_config()
manifest_path = cfg.manifest_path
if not manifest_path.exists():
    print('No manifest found — generating a synthetic smoke slice for this demo.')
    os.environ.setdefault('DERMAFACE_DATA_DIR', str(ROOT / 'data_smoke'))
    from make_smoke_slice import make_slice
    from dermaface.data import manifest as M, splits as S
    cfg = load_config()
    make_slice(cfg.data_dir / 'raw', per_class=12)
    manifest_path, _ = M.build_manifest(cfg.data_dir / 'raw', require_image_exists=True)
    S.make_splits(manifest_path)
df = pd.read_csv(manifest_path)
print(f'{len(df)} rows from {manifest_path}')
df.head()

## 1. Class balance

Rosacea is expected to be rare — this is why the requirements use **macro-F1** and a **per-class recall floor**, and why the train loader uses a weighted sampler.

In [ ]:
counts = df['label'].value_counts().reindex(CLASS_NAMES).fillna(0).astype(int)
display(counts.to_frame('n').assign(pct=lambda d: (100*d['n']/d['n'].sum()).round(1)))
ax = counts.plot(kind='bar', color='#4C72B0', figsize=(6,3.5))
ax.set_title('Class balance'); ax.set_ylabel('images'); ax.set_xlabel('')
plt.tight_layout(); plt.show()
imb = counts.max() / max(counts[counts>0].min(), 1)
print(f'Imbalance ratio (largest/smallest non-empty class): {imb:.1f}x')

## 2. Fitzpatrick skin-type distribution

Fairness target Fa1 needs enough samples in **every** skin-type group. Sparse dark-skin cells (V–VI) are the usual risk in dermatology datasets.

In [ ]:
order = FITZPATRICK_TYPES + ['unknown']
st = df['skin_type'].value_counts().reindex(order).fillna(0).astype(int)
ax = st.plot(kind='bar', color='#55A868', figsize=(6,3.5))
ax.set_title('Fitzpatrick skin-type distribution'); ax.set_ylabel('images'); ax.set_xlabel('')
plt.tight_layout(); plt.show()
display(st.to_frame('n'))

### 2a. Class × skin-type (fairness cross-tab)

Cells that are empty or tiny here are where the model will be least reliable and where the fairness gap will be hardest to measure.

In [ ]:
ct = pd.crosstab(df['label'], df['skin_type']).reindex(index=CLASS_NAMES).reindex(columns=order, fill_value=0)
plt.figure(figsize=(7,3.5))
sns.heatmap(ct, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title('Class x Fitzpatrick skin type'); plt.ylabel(''); plt.xlabel('')
plt.tight_layout(); plt.show()
thin = ct.where(ct>0).stack().pipe(lambda s: s[s < 5])
print('Under-populated (class, skin_type) cells (<5):', dict(thin) if len(thin) else 'none')

## 3. Source & split composition

Confirms all three datasets are represented and that the stratified split preserved the class/skin-type proportions (test set frozen).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10,3.5))
df['source'].value_counts().plot(kind='bar', ax=axes[0], color='#C44E52')
axes[0].set_title('images per source'); axes[0].set_xlabel('')
if 'split' in df and df['split'].nunique() > 1:
    pd.crosstab(df['label'], df['split']).reindex(CLASS_NAMES).plot(kind='bar', stacked=True, ax=axes[1])
    axes[1].set_title('class x split'); axes[1].set_xlabel('')
plt.tight_layout(); plt.show()
if 'split' in df:
    print(df['split'].value_counts().to_dict())

## 4. Image quality

Samples up to 200 images and measures resolution, brightness, and a blur score (variance of a Laplacian filter — lower = blurrier). These drive the cleaning step (drop corrupt / low-quality / blurry) before training.

In [ ]:
def laplacian_var(gray: np.ndarray) -> float:
    k = np.array([[0,1,0],[1,-4,1],[0,1,0]], dtype=np.float32)
    from numpy.lib.stride_tricks import sliding_window_view as swv
    if gray.shape[0] < 3 or gray.shape[1] < 3:
        return float('nan')
    win = swv(gray, (3,3))
    conv = (win * k).sum(axis=(-1,-2))
    return float(conv.var())

rows = []
sample = df.sample(min(200, len(df)), random_state=0)
for p in sample['path']:
    fp = cfg.data_dir / p
    if not fp.exists():
        continue
    try:
        with Image.open(fp) as im:
            im = im.convert('RGB'); w, h = im.size
            g = np.asarray(im.convert('L'), dtype=np.float32)
        rows.append({'w': w, 'h': h, 'mp': (w*h)/1e6, 'brightness': float(g.mean()), 'blur': laplacian_var(g)})
    except Exception as e:
        rows.append({'w': None, 'h': None, 'error': str(e)})
q = pd.DataFrame(rows)
IMAGES_ON_DISK = ('w' in q.columns) and (q['w'].notna().any())
if not IMAGES_ON_DISK:
    print(f'No image files found on disk for the {len(sample)} sampled rows.')
    print('This is a metadata-only manifest — image-quality checks are skipped.')
    print('Download images (download_images=True) to enable this section.')
else:
    print(f'measured {len(q)} images; {int(q["w"].isna().sum())} unreadable')
    display(q.select_dtypes('number').describe().round(2))

In [ ]:
qn = q.dropna(subset=['w']) if ('w' in q.columns) else q.iloc[0:0]
if len(qn):
    fig, ax = plt.subplots(1, 3, figsize=(12,3.2))
    qn['mp'].plot(kind='hist', bins=20, ax=ax[0], color='#8172B3'); ax[0].set_title('resolution (megapixels)')
    qn['brightness'].plot(kind='hist', bins=20, ax=ax[1], color='#CCB974'); ax[1].set_title('mean brightness (0-255)')
    qn['blur'].plot(kind='hist', bins=20, ax=ax[2], color='#64B5CD'); ax[2].set_title('blur score (Laplacian var)')
    plt.tight_layout(); plt.show()
    blur_thr = qn['blur'].quantile(0.05)
    dark = qn[(qn['brightness']<40)|(qn['brightness']>230)]
    print(f'suggested blur cutoff (5th pct): {blur_thr:.1f} -> {int((qn["blur"]<blur_thr).sum())} flagged')
    print(f'over/under-exposed (bright<40 or >230): {len(dark)}')

## 5. Label map (harmonization crosswalk)

`data/external/label_map.csv` records how each raw source condition maps to the four classes (or is dropped). This is the audit trail for the manifest's labels and the starting point for Notebook 02.

In [ ]:
from dermaface.data import manifest as M
lm_path = cfg.label_map_path
if not lm_path.exists():
    M.build_label_map(cfg.data_dir / 'raw')
lm = pd.read_csv(lm_path)
print('distinct raw labels:', len(lm), '| kept:', (lm['action']=='kept').sum(), '| dropped:', (lm['action']=='dropped').sum())
print('\nTop KEPT mappings:')
display(lm[lm['action']=='kept'].sort_values('n_rows', ascending=False).head(10))
print('Largest DROPPED groups (candidates for Notebook 02 to reclaim):')
display(lm[lm['action']=='dropped'].sort_values('n_rows', ascending=False).head(10))

## 6. Data QA report

`make qa` writes `data/processed/qa/data_quality_report.csv`, flagging corrupt / duplicate / oversized / undersized images and missing label / skin-type fields. Before images are downloaded it still flags missing images and unknown skin types.

In [ ]:
from dermaface.data import qa as QA
report_path, qa_summary = QA.run_qa(manifest_path)
print(QA.render_summary(qa_summary))
qadf = pd.read_csv(report_path)
flagged = qadf[qadf['issues'].fillna('') != '']
print(f'\n{len(flagged)}/{len(qadf)} rows have at least one flag. Sample:')
display(flagged[['path','source','label','skin_type','split','issues']].head(8))

## 7. Summary & next steps

**What we see (on real data, fill in the numbers):**
- Class balance and the rosacea rarity that justifies macro-F1 + weighted sampling.
- Skin-type coverage; flag any V/VI sparsity that threatens the fairness target.
- Image-quality spread; set concrete cleaning thresholds (blur, exposure, min-resolution).

**Next:**
- `02_label_harmonization.ipynb` — finalize the taxonomy + severity method (Aparna + Iva).
- Wire `preprocessing.has_face` into cleaning; dedup with perceptual hashing (imagehash).
- Baseline transfer-learning run once the manifest is on real images.

---
### Member contributions (Sprint 1, data)
- **Aparna** — data strategy, source selection & licensing, label taxonomy, EDA.
- **Rolando** — download pipeline, manifest builder, stratified splits, dataloaders, QA/tests.